# 08 - The transaction model from the source code

Notebook 07 showed that a model on single transactions, with the noisy-or of an account's transactions as the account score, scores higher than the account model. The project now uses it. This notebook checks that `src` reproduces notebook 07, sets the flagging threshold, and asks what the score and its explanation can be trusted for.

**The threshold rule is fixed here:** in every round, the lowest threshold with at least 0.7 precision on the validation accounts. The account model gets the same rule for the comparison.

**Limit carried over from notebook 07.** The idea came from errors seen on all folds, and no untouched data set is available. Nothing here removes that bias. Test scores below do not change any choice.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.explain import shap_values
from skeptical_second_reader.features import ALL_FEATURES
from skeptical_second_reader.model import fit_final, precision_threshold
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    split_groups,
)
from skeptical_second_reader.transactions import (
    TX_FEATURES,
    account_positions,
    explain_strongest,
    strongest_transaction,
    transaction_cv,
    transaction_features,
    transaction_labels,
    transaction_table,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)

X = pd.read_parquet(DATA_INTERIM / "experiment_features.parquet")[ALL_FEATURES]
y = labels.loc[X.index]
fold_of = cached_folds(labels, groups, DATA_INTERIM).loc[X.index]

table = transaction_table(df)
features = transaction_features(table)
tx_labels = transaction_labels(df)
sender, receiver = account_positions(table, X.index)
print(
    f"accounts {len(X):,}  positives {y.sum():,}  "
    f"transactions {len(table):,}  labelled {tx_labels.sum():,}  "
    f"features {features.shape[1]}"
)

accounts 515,078  positives 6,347  transactions 5,077,237  labelled 4,522  features 21
CPU times: total: 56 s
Wall time: 53.9 s


## 1. Does the source code reproduce notebook 07?

The per-round test AUC-PR of notebook 07 is written down here, and the account scores are compared with the file notebook 07 saved.

In [2]:
%%time


def per_round(scores, target=y, fold_labels=fold_of):
    rows = []
    for k in range(N_FOLDS):
        test = fold_labels == k
        rows.append(
            {
                "auc_pr": average_precision_score(target[test], scores[test]),
                "auc_roc": roc_auc_score(target[test], scores[test]),
            }
        )
    return pd.DataFrame(rows)


def mean_sd(values):
    return f"{values.mean():.4f} +/- {values.std(ddof=1):.4f}"


result = transaction_cv(features, tx_labels, table, y, fold_of, precision_threshold)
tx_rounds = per_round(result.scores)
NOTEBOOK_07 = [0.5016, 0.4618, 0.4689, 0.4560, 0.5146]
assert tx_rounds["auc_pr"].round(4).tolist() == NOTEBOOK_07
stored = pd.read_parquet(DATA_INTERIM / "tx_model_scores.parquet")
assert stored.index.equals(result.scores.index)
largest_gap = (stored["tx_noisy_or"] - result.scores).abs().max()
assert largest_gap < 1e-6
print("per-round test AUC-PR equals notebook 07: True")
print("largest score difference from notebook 07:", float(largest_gap))
print("transaction model AUC-PR:", mean_sd(tx_rounds["auc_pr"]))
print("transaction model AUC-ROC:", mean_sd(tx_rounds["auc_roc"]))

final = cv_predict(X, y, fold_of, fit_final, precision_threshold)
final_rounds = per_round(final.scores)
assert final_rounds["auc_pr"].round(4).tolist() == [
    0.3812,
    0.3451,
    0.3552,
    0.3601,
    0.3777,
]
print("account model AUC-PR:", mean_sd(final_rounds["auc_pr"]))
print("account model AUC-ROC:", mean_sd(final_rounds["auc_roc"]))
gain = tx_rounds["auc_pr"] - final_rounds["auc_pr"]
print("gain over the account model:", mean_sd(gain))

per-round test AUC-PR equals notebook 07: True
largest score difference from notebook 07: 0.0
transaction model AUC-PR: 0.4806 +/- 0.0260
transaction model AUC-ROC: 0.9257 +/- 0.0081


account model AUC-PR: 0.3639 +/- 0.0153
account model AUC-ROC: 0.9109 +/- 0.0072
gain over the account model: 0.1167 +/- 0.0147
CPU times: total: 57min 22s
Wall time: 4min 28s


## 2. The threshold

Both models flag at the lowest threshold with at least 0.7 validation precision, chosen in each round without the test accounts. The second table is a diagnostic, not a result: the widest set of test accounts with precision of at least 0.7 and at least 0.5, chosen with the test labels, as a best case to compare with.

In [3]:
def operating(flags):
    rows = []
    for k in range(N_FOLDS):
        test = fold_of == k
        chosen = flags[test] == 1
        hits = y[test][chosen].sum()
        rows.append(
            {
                "round": k,
                "flagged": int(chosen.sum()),
                "true_positives": int(hits),
                "false_positives": int(chosen.sum() - hits),
                "precision": hits / max(chosen.sum(), 1),
                "recall": hits / y[test].sum(),
            }
        )
    return pd.DataFrame(rows)


for name, flags in {
    "transaction model": result.flags,
    "account model": final.flags,
}.items():
    table_of = operating(flags)
    print(name)
    print(table_of.round(4).to_string(index=False))
    print(
        "total flagged",
        table_of["flagged"].sum(),
        " true positives",
        table_of["true_positives"].sum(),
        " false positives",
        table_of["false_positives"].sum(),
        " precision",
        mean_sd(table_of["precision"]),
        " recall",
        mean_sd(table_of["recall"]),
    )
    print()
print("thresholds of the transaction model:", np.round(result.thresholds, 4))
print("thresholds of the account model:", np.round(final.thresholds, 4))

rows = []
for name, scores in {
    "account model": stored["final"],
    "transaction model": result.scores,
}.items():
    for wanted in [0.7, 0.5]:
        hits = misses = 0
        for k in range(N_FOLDS):
            test = fold_of == k
            ranked = y[test].to_numpy()[
                np.argsort(-scores[test].to_numpy(), kind="stable")
            ]
            found = ranked.cumsum()
            precision = found / np.arange(1, len(found) + 1)
            n = np.flatnonzero(precision >= wanted).max() + 1
            hits += found[n - 1]
            misses += n - found[n - 1]
        rows.append(
            {
                "model": name,
                "precision_at_least": wanted,
                "true_positives": hits,
                "false_positives": misses,
                "recall": hits / y.sum(),
            }
        )
print("best case, chosen on test labels (diagnostic only)")
print(pd.DataFrame(rows).round(4).to_string(index=False))

transaction model


 round  flagged  true_positives  false_positives  precision  recall
     0      757             529              228     0.6988  0.4165
     1      694             475              219     0.6844  0.3743
     2      684             470              214     0.6871  0.3704
     3      689             459              230     0.6662  0.3617
     4      748             525              223     0.7019  0.4134
total flagged 3572  true positives 2458  false positives 1114  precision 0.6877 +/- 0.0141  recall 0.3873 +/- 0.0257

account model
 round  flagged  true_positives  false_positives  precision  recall
     0      483             346              137     0.7164  0.2724
     1      447             303              144     0.6779  0.2388
     2      451             322              129     0.7140  0.2537
     3      484             339              145     0.7004  0.2671
     4      474             341              133     0.7194  0.2685
total flagged 2339  true positives 1651  false posit

best case, chosen on test labels (diagnostic only)
            model  precision_at_least  true_positives  false_positives  recall
    account model                 0.7            1647              705  0.2595
    account model                 0.5            1864             1864  0.2937
transaction model                 0.7            2451             1048  0.3862
transaction model                 0.5            2612             2612  0.4115


## 3. Which accounts change?

The groups of notebook 06, at the threshold of section 2. `conversion rule` is exactly one conversion with an ACH send in the same minute. Accounts in the groups overlap with the rule.

In [4]:
in_window_label = tx_labels.to_numpy() == 1
is_self = sender == receiver
is_ach = (table["payment_format"] == "ACH").to_numpy()


def touches(mask):
    counted = np.zeros(len(X), dtype="int64")
    np.add.at(counted, sender[mask], 1)
    np.add.at(counted, receiver[mask & ~is_self], 1)
    return pd.Series(counted, index=X.index)


bad_count = touches(in_window_label)
ach_bad_count = touches(in_window_label & is_ach)
positive = y == 1
converted = is_self & features["changes_currency"].to_numpy().astype(bool)
conversions = touches(converted)
after_conversion = features["after_conversion"].to_numpy() == 1
sends_after = np.zeros(len(X), dtype=bool)
sends_after[sender[after_conversion & is_ach & ~is_self]] = True
rule = pd.Series(sends_after, index=X.index) & (conversions == 1)
print("conversion rule accounts:", int(rule.sum()), " positives:", int(y[rule].sum()))

report_groups = {
    "all positives": positive,
    "one ACH label, one currency": positive
    & (ach_bad_count > 0)
    & (bad_count == 1)
    & (X["n_currencies"] == 1),
    "labelled, none ACH": positive & (bad_count > 0) & (ach_bad_count == 0),
    "no label in the window": positive & (bad_count == 0),
    "2+ labels, some ACH": positive & (ach_bad_count > 0) & (bad_count >= 2),
    "conversion rule": positive & rule,
}
rows = []
for name, mask in report_groups.items():
    rows.append(
        {
            "group": name,
            "positives": int(mask.sum()),
            "account model": final.flags[mask].mean(),
            "transaction model": result.flags[mask].mean(),
        }
    )
print("recall per group, all rounds together")
print(pd.DataFrame(rows).round(4).to_string(index=False))

rows = []
for name, part in {"inside the rule": rule, "outside the rule": ~rule}.items():
    for model, flags, scores in [
        ("account model", final.flags, final.scores),
        ("transaction model", result.flags, result.scores),
    ]:
        chosen = part & (flags == 1)
        rows.append(
            {
                "accounts": name,
                "model": model,
                "flagged": int(chosen.sum()),
                "precision": y[chosen].mean(),
                "recall": y[chosen].sum() / y[part].sum(),
                "auc_pr": per_round(scores[part], y[part], fold_of[part])[
                    "auc_pr"
                ].mean(),
            }
        )
print()
print(pd.DataFrame(rows).round(4).to_string(index=False))

conversion rule accounts: 1187  positives: 1097


recall per group, all rounds together
                      group  positives  account model  transaction model
              all positives       6347         0.2601             0.3873
one ACH label, one currency       3457         0.0735             0.2627
         labelled, none ACH        740         0.0230             0.0257
     no label in the window        485         0.0021             0.0021
        2+ labels, some ACH        986         0.7728             0.8986
            conversion rule       1097         0.9927             0.9973



        accounts             model  flagged  precision  recall  auc_pr
 inside the rule     account model     1114     0.9776  0.9927  0.9986
 inside the rule transaction model     1127     0.9707  0.9973  0.9962
outside the rule     account model     1225     0.4588  0.1070  0.1720
outside the rule transaction model     2445     0.5579  0.2598  0.3386


## 4. Does the score simply follow activity?

A noisy-or rises with every further transaction, so busy accounts could score high for no other reason. Three checks: accounts by number of transactions, a score that is only the number of transactions, and a model trained on shuffled transaction labels, inside and outside the conversion rule.

In [5]:
%%time
activity = touches(np.ones(len(table), dtype=bool))
bins = pd.cut(
    activity,
    [0, 1, 2, 5, 20, 100, np.inf],
    labels=["1", "2", "3-5", "6-20", "21-100", "101+"],
)
by_activity = pd.DataFrame(
    {
        "accounts": activity.groupby(bins, observed=True).size(),
        "positive_rate": y.groupby(bins, observed=True).mean(),
        "mean_score": result.scores.groupby(bins, observed=True).mean(),
        "flagged": result.flags.groupby(bins, observed=True).sum(),
        "precision": y[result.flags == 1]
        .groupby(bins[result.flags == 1], observed=True)
        .mean(),
        "recall": result.flags[positive].groupby(bins[positive], observed=True).mean(),
    }
)
print(by_activity.round(4).to_string())
negatives = y == 0
print(
    "Spearman, score and transactions, negatives:",
    round(result.scores[negatives].corr(activity[negatives], method="spearman"), 3),
)
print(
    "Spearman, score and transactions, positives:",
    round(result.scores[positive].corr(activity[positive], method="spearman"), 3),
)

rng = np.random.default_rng(0)
shuffled_labels = pd.Series(
    rng.permutation(tx_labels.to_numpy()), index=tx_labels.index
)
shuffled = transaction_cv(features, shuffled_labels, table, y, fold_of, lambda *_: 0.5)
rows = []
for name, scores in {
    "number of transactions": activity.astype("float64"),
    "shuffled labels": shuffled.scores,
    "transaction model": result.scores,
}.items():
    rows.append(
        {
            "score": name,
            "all": mean_sd(per_round(scores)["auc_pr"]),
            "outside the rule": mean_sd(
                per_round(scores[~rule], y[~rule], fold_of[~rule])["auc_pr"]
            ),
        }
    )
print()
print(pd.DataFrame(rows).to_string(index=False))
print(
    "base rate, all:",
    round(y.mean(), 4),
    " outside the rule:",
    round(y[~rule].mean(), 4),
)

        accounts  positive_rate  mean_score  flagged  precision  recall
1          90862         0.0012      0.0008       12     1.0000  0.1101
2         106503         0.0025      0.0024      113     0.6549  0.2772
3-5        78240         0.0141      0.0145      616     0.7792  0.4352
6-20       76436         0.0223      0.0245      883     0.6116  0.3162
21-100    154694         0.0185      0.0194     1685     0.6748  0.3980
101+        8343         0.0363      0.0359      263     0.8175  0.7096


Spearman, score and transactions, negatives: 0.624
Spearman, score and transactions, positives: 0.152



                 score               all  outside the rule
number of transactions 0.0264 +/- 0.0080 0.0222 +/- 0.0065
       shuffled labels 0.0263 +/- 0.0077 0.0222 +/- 0.0065
     transaction model 0.4806 +/- 0.0260 0.3386 +/- 0.0369
base rate, all: 0.0123  outside the rule: 0.0102
CPU times: total: 15min 27s
Wall time: 1min 32s


## 5. What if `pair_count` could not look ahead?

`pair_count` counts the pair's transactions in the whole window, also those after the one it describes. Here it counts only those up to and including the transaction itself, so a pair's first transaction always has 1. All other features still use the whole window, so this removes one look-ahead, not all of them.

In [6]:
%%time
ordered = table.sort_values("timestamp", kind="stable")
so_far = (ordered.groupby(["sender", "receiver"]).cumcount() + 1).sort_index()
past_features = features.assign(pair_count=so_far.to_numpy(dtype="float32"))
past = transaction_cv(past_features, tx_labels, table, y, fold_of, precision_threshold)
past_rounds = per_round(past.scores)
print("AUC-PR, pair count without look-ahead:", mean_sd(past_rounds["auc_pr"]))
print(
    "against the transaction model:",
    mean_sd(past_rounds["auc_pr"] - tx_rounds["auc_pr"]),
)
print(
    "against the account model:",
    mean_sd(past_rounds["auc_pr"] - final_rounds["auc_pr"]),
)
print(
    "outside the rule:",
    mean_sd(per_round(past.scores[~rule], y[~rule], fold_of[~rule])["auc_pr"]),
)
past_table = operating(past.flags)
print(
    "flagged",
    past_table["flagged"].sum(),
    " true positives",
    past_table["true_positives"].sum(),
    " false positives",
    past_table["false_positives"].sum(),
    " recall",
    mean_sd(past_table["recall"]),
)
print(
    "one ACH label, one currency, recall:",
    round(past.flags[report_groups["one ACH label, one currency"]].mean(), 4),
)

AUC-PR, pair count without look-ahead: 0.4623 +/- 0.0262
against the transaction model: -0.0182 +/- 0.0080
against the account model: 0.0985 +/- 0.0147


outside the rule: 0.3163 +/- 0.0385


flagged 3403  true positives 2359  false positives 1044  recall 0.3717 +/- 0.0230
one ACH label, one currency, recall: 0.2421
CPU times: total: 41min 53s
Wall time: 3min 27s


## 6. Calibration

The LLM will read the score. Does a score of 0.8 mean that about 80% of such accounts are positive? Out-of-fold scores, all rounds together.

In [7]:
bins = [0, 0.01, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0]
binned = pd.cut(result.scores, bins, include_lowest=True)
calibration = pd.DataFrame(
    {
        "accounts": result.scores.groupby(binned, observed=False).size(),
        "mean_score": result.scores.groupby(binned, observed=False).mean(),
        "positive_rate": y.groupby(binned, observed=False).mean(),
    }
)
print(calibration.round(4).to_string())

                accounts  mean_score  positive_rate
score                                              
(-0.001, 0.01]    431808      0.0011         0.0019
(0.01, 0.05]       60013      0.0229         0.0276
(0.05, 0.1]        12848      0.0697         0.0627
(0.1, 0.2]          5610      0.1353         0.0834
(0.2, 0.4]          1826      0.2738         0.1046
(0.4, 0.6]           443      0.4845         0.2348
(0.6, 0.8]           238      0.6895         0.3739
(0.8, 0.9]           103      0.8497         0.5922
(0.9, 1.0]          2189      0.9956         0.9740


## 7. Does the explanation describe the account?

The LLM will be shown an account's strongest transaction and its SHAP values, from the model of the round in which that account was tested. SHAP explains the transaction's score, not the account's noisy-or, so the first question is how much of a flagged account's score that transaction carries. Then: is it a labelled transaction, and which features carry it? A negative account has no labelled transaction by definition, so the labelled share is shown for true positives only.

In [8]:
%%time
explained = explain_strongest(result, features, fold_of)
strongest_rows = result.strongest.to_numpy()
strongest_score = pd.Series(np.nan, index=X.index, name="strongest_score")
for k, model in enumerate(result.models):
    part = fold_of == k
    strongest_score[part] = model.predict_proba(features.iloc[strongest_rows[part]])[
        :, 1
    ]

threshold_of = fold_of.map(dict(enumerate(result.thresholds)))
flagged = result.flags == 1
outcomes = {
    "true positives": flagged & positive,
    "false positives": flagged & ~positive,
}
strongest_label = pd.Series(tx_labels.to_numpy()[strongest_rows], index=X.index)
strongest_format = pd.Series(
    table["payment_format"].to_numpy()[strongest_rows], index=X.index
)
rows = []
for name, mask in outcomes.items():
    rows.append(
        {
            "outcome": name,
            "accounts": int(mask.sum()),
            "median strongest / account score": (
                strongest_score[mask] / result.scores[mask]
            ).median(),
            "strongest alone reaches the threshold": (
                strongest_score[mask] >= threshold_of[mask]
            ).mean(),
            "strongest is labelled": strongest_label[mask].mean(),
            "strongest is ACH": (strongest_format[mask] == "ACH").mean(),
        }
    )
print(pd.DataFrame(rows).round(4).T.to_string(header=False))

contributions = explained[TX_FEATURES]
largest = contributions[flagged].abs().idxmax(axis=1)
print()
print("largest contribution of the strongest transaction, flagged accounts")
print(largest.value_counts(normalize=True).head(8).round(3).to_string())
order = np.argsort(-contributions[flagged].abs().to_numpy(), axis=1)[:, :3]
in_top_3 = (
    pd.Series(np.array(TX_FEATURES)[order].ravel()).value_counts() / flagged.sum()
)
print()
print("share of flagged accounts with the feature among the top 3")
print(in_top_3.head(8).round(3).to_string())

outcome                                true positives  false positives
accounts                                         2458             1114
median strongest / account score                  1.0           0.9585
strongest alone reaches the threshold          0.9788           0.6885
strongest is labelled                          0.9788              0.0
strongest is ACH                               0.9943           0.9713

largest contribution of the strongest transaction, flagged accounts
pair_count                    0.567
payment_format                0.423
sender_tx                     0.003
log_usd_over_sender_median    0.001
after_conversion              0.001
receiver_tx                   0.001
receiver_in_degree            0.001
paid_currency                 0.001

share of flagged accounts with the feature among the top 3
pair_count            0.933
payment_format        0.855
after_conversion      0.540
log_usd               0.247
sender_out_degree     0.132
sender_tx        

## 8. Are the explanations stable?

All five models look at the same flagged accounts. The models share training data, and four of them trained on transactions of these accounts or of their neighbours, so this measures stability under a changed training set, not independence.

In [9]:
%%time
flagged_accounts = X.index[flagged]
position = X.index.get_indexer(flagged_accounts)
near = np.isin(sender, position) | np.isin(receiver, position)
near_rows = np.flatnonzero(near)
near_features = features.iloc[near_rows]
chosen_by_model = []
for model in result.models:
    tx_scores = model.predict_proba(near_features)[:, 1]
    top = strongest_transaction(tx_scores, sender[near], receiver[near], len(X))
    chosen_by_model.append(near_rows[top[position]])

same_row = [
    (chosen_by_model[i] == chosen_by_model[j]).mean()
    for i in range(N_FOLDS)
    for j in range(i + 1, N_FOLDS)
]
print(
    f"same strongest transaction between two models: "
    f"mean {np.mean(same_row):.3f}, lowest pair {np.min(same_row):.3f}"
)

rows_of_flagged = result.strongest[flagged_accounts].to_numpy()
per_model = [
    shap_values(model, features.iloc[rows_of_flagged])[TX_FEATURES]
    for model in result.models
]
top_sets = [
    [set(row) for row in np.argsort(-v.abs().to_numpy(), axis=1)[:, :3]]
    for v in per_model
]
overlaps = [
    np.mean([len(p & q) / 3 for p, q in zip(top_sets[i], top_sets[j], strict=True)])
    for i in range(N_FOLDS)
    for j in range(i + 1, N_FOLDS)
]
print(
    f"top-3 feature overlap on that transaction between two models: "
    f"mean {np.mean(overlaps):.3f}, lowest pair {np.min(overlaps):.3f}"
)

same strongest transaction between two models: mean 0.869, lowest pair 0.846


top-3 feature overlap on that transaction between two models: mean 0.790, lowest pair 0.695
CPU times: total: 49.8 s
Wall time: 3.86 s


## 9. Files for the next step, and candidate cases

Out-of-fold scores, flags, each account's strongest transaction and its SHAP values go to `data/interim`, which is not in the repository, for the interface and the LLM step. `strongest_row` is a row of `transaction_table`. Candidate cases are drawn at random from five kinds of model outcome. They are a proposal for the first slice, not the experiment's final sample. Only counts and score ranges are shown here.

In [10]:
results = pd.DataFrame(
    {
        "score": result.scores,
        "threshold": threshold_of,
        "flag": result.flags,
        "label": y,
        "fold": fold_of,
        "strongest_row": result.strongest,
        "strongest_score": strongest_score,
    }
)
results.to_parquet(DATA_INTERIM / "transaction_scores.parquet")
explained.to_parquet(DATA_INTERIM / "transaction_shap.parquet")


def logit(values):
    clipped = values.clip(1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped))


distance = (logit(results["score"]) - logit(results["threshold"])).abs()
pools = {
    "confident true positive": results[
        results["flag"].eq(1) & results["label"].eq(1)
    ].nlargest(500, "score"),
    "confident false positive": results[
        results["flag"].eq(1) & results["label"].eq(0)
    ].nlargest(100, "score"),
    "missed positive": results[
        results["flag"].eq(0) & results["label"].eq(1)
    ].nsmallest(500, "score"),
    "clear negative": results[results["flag"].eq(0) & results["label"].eq(0)].nsmallest(
        5000, "score"
    ),
    "near the threshold": results.loc[distance.nsmallest(200).index],
}
cases = pd.concat(
    [pool.sample(2, random_state=0).assign(kind=kind) for kind, pool in pools.items()]
)
cases.to_parquet(DATA_INTERIM / "transaction_candidate_cases.parquet")
print(
    cases.groupby("kind", sort=False)
    .agg(
        cases=("score", "size"),
        lowest_score=("score", "min"),
        highest_score=("score", "max"),
        positives=("label", "sum"),
        flagged=("flag", "sum"),
    )
    .round(4)
    .to_string()
)

                          cases  lowest_score  highest_score  positives  flagged
kind                                                                            
confident true positive       2        1.0000         1.0000          2        2
confident false positive      2        0.8290         0.9950          0        2
missed positive               2        0.0000         0.0006          2        0
clear negative                2        0.0000         0.0000          0        0
near the threshold            2        0.2612         0.3738          0        0
